# Instacart — Analysis Tables and KPI Creation

This notebook uses the cleaned data stored in `workspace.cleaned_data` to create analysis tables and KPIs.

The main goals are to:

- analyze customer activity
- measure product performance
- study reorder behavior
- analyze cart composition
- study orders by day and hour
- create summary tables for analysis and dashboards

The results will be saved in:

`workspace.analysis_data`

## 1. Set Up the Analysis Environment

This step defines the main project settings and creates the `analysis_data` schema.

The schema will store the KPI and analysis tables created in this notebook.

In [0]:

from pyspark.sql import functions as F

catalog_name = "workspace"
cleaned_schema_name = "cleaned_data"
analysis_schema_name = "analysis_data"

# Create the schema used for analysis tables
spark.sql(
    f"""
    CREATE SCHEMA IF NOT EXISTS
    {catalog_name}.{analysis_schema_name}
    """
)

print(
    f"the schema {catalog_name}.{analysis_schema_name} "
    "is ready."
)

the schema workspace.analysis_data is ready.


### 1.1 Check the Analysis Schema

This check confirms that the `analysis_data` schema was created successfully in the `workspace` catalog.

The result should show:

`analysis_data`

In [0]:

display(
    spark.sql("SHOW SCHEMAS IN workspace")
    .filter(F.col("databaseName") == "analysis_data")
)

databaseName
analysis_data


## 2. Load the Cleaned Tables

This step loads the cleaned tables from the `cleaned_data` layer so they can be used to create KPIs and analysis tables.

The following tables are loaded:

- `departments`
- `aisles`
- `products`
- `orders`
- `order_products`

In [0]:


departments_cleaned_df = spark.table(
    "workspace.cleaned_data.departments"
)

aisles_cleaned_df = spark.table(
    "workspace.cleaned_data.aisles"
)

products_cleaned_df = spark.table(
    "workspace.cleaned_data.products"
)

orders_cleaned_df = spark.table(
    "workspace.cleaned_data.orders"
)

order_products_cleaned_df = spark.table(
    "workspace.cleaned_data.order_products"
)

print("All cleaned tables loaded successfully.")

All cleaned tables loaded successfully.


## 3. Create Order-Level KPIs

This step creates one summary row for each order by using the `order_products` table.

For each order, the following KPIs are calculated:

- `basket_size`: total number of products in the order
- `distinct_product_count`: number of different products
- `reordered_product_count`: number of products that were ordered before
- `new_product_count`: number of new products
- `reorder_rate`: share of reordered products in the basket

The result is then joined with the `orders` table to add customer and time information.

Only `prior` and `train` orders are included because these orders have product-level details.

In [0]:



order_product_summary_df = (
    order_products_cleaned_df
    .groupBy(
        "order_id",
        "order_set"
    )
    .agg(
        # Total number of products in the order
        F.count("*").alias("basket_size"),

         # Number of different products
        F.countDistinct("product_id").alias(
            "distinct_product_count"
        ),

        # Number of products ordered before
        F.sum(
            F.col("is_reordered").cast("int")
        ).alias("reordered_product_count")
    )

    # Number of new products
    .withColumn(
        "new_product_count",
        F.col("basket_size")
        - F.col("reordered_product_count")
    )

    # Share of reordered products in the basket
    .withColumn(
        "reorder_rate",
        F.round(
            F.col("reordered_product_count")
            / F.col("basket_size"),
            4
        )
    )
)


# Add customer and time information from the orders table
orders_with_kpis_df = (
    orders_cleaned_df

    # Only prior and train orders have product details
    .filter(
        F.col("eval_set").isin("prior", "train")
    )

    .join(
        order_product_summary_df,
        on="order_id",
        how="inner"
    )

    .select(
        "order_id",
        "user_id",
        "eval_set",
        "order_set",
        "order_number",
        "order_dow",
        "order_hour_of_day",
        "order_time_period",
        "days_since_prior_order",
        "is_first_order",
        "is_final_order",
        "basket_size",
        "distinct_product_count",
        "reordered_product_count",
        "new_product_count",
        "reorder_rate"
    )
)

print("Order-level KPIs created successfully.")

Order-level KPIs created successfully.


### 3.1 Validate the Order-Level KPIs

This check verifies that the order-level KPIs were created correctly.

The validation checks:

- the total number of orders
- whether each `order_id` appears only once
- whether any basket has 0 or fewer products
- whether the number of distinct products is greater than the basket size
- whether the number of reordered products is greater than the basket size
- whether the number of new products is correct
- whether `reorder_rate` is between 0 and 1

All error counts should be equal to 0.

In [0]:
# Validate the order-level KPIs

orders_kpis_quality_df = (
    orders_with_kpis_df
    .agg(
        # Total number of orders
        F.count("*").alias("total_orders"),

        # Number of distinct orders
        F.countDistinct("order_id").alias(
            "distinct_orders"
        ),

        # Basket size should be greater than 0
        F.sum(
            F.when(
                F.col("basket_size") <= 0,
                1
            ).otherwise(0)
        ).alias("invalid_basket_sizes"),

        # Distinct product count cannot exceed basket size
        F.sum(
            F.when(
                F.col("distinct_product_count")
                > F.col("basket_size"),
                1
            ).otherwise(0)
        ).alias("invalid_distinct_product_counts"),

        # Reordered and new products should add up to basket size
        F.sum(
            F.when(
                F.col("reordered_product_count")
                + F.col("new_product_count")
                != F.col("basket_size"),
                1
            ).otherwise(0)
        ).alias("inconsistent_basket_breakdown"),

        # Reorder rate should be between 0 and 1
        F.sum(
            F.when(
                ~F.col("reorder_rate").between(0, 1),
                1
            ).otherwise(0)
        ).alias("invalid_reorder_rates"),

        # eval_set should match the order-product source
        F.sum(
            F.when(
                F.col("eval_set") != F.col("order_set"),
                1
            ).otherwise(0)
        ).alias("inconsistent_order_source")
    )
)

display(orders_kpis_quality_df)

total_orders,distinct_orders,invalid_basket_sizes,invalid_distinct_product_counts,inconsistent_basket_breakdown,invalid_reorder_rates,inconsistent_order_source
3346083,3346083,0,0,0,0,0


## 4. Create Customer-Level KPIs

This step creates one summary row for each customer.

The customer KPIs include:

- total number of orders
- number of orders with product details
- total number of products purchased
- average basket size
- number of reordered products
- number of new products
- overall reorder rate
- average number of days between orders
- type of final order
- customer frequency group

These KPIs will help us understand customer activity and buying behavior.


In [0]:
# 1. General order activity for each customer

customer_order_activity_df = (
    orders_cleaned_df
    .groupBy("user_id")
    .agg(
        # Total number of orders, including the final test order
        F.count("*").alias("total_order_count"),

        # Highest order number for the customer
        F.max("order_number").alias("max_order_number"),

        # Average number of days between orders
        # The null value of the first order is ignored by avg()
        F.round(
            F.avg("days_since_prior_order"),
            2
        ).alias("avg_days_between_orders"),

        # Type of the final order: train or test
        F.max(
            F.when(
                F.col("is_final_order"),
                F.col("eval_set")
            )
        ).alias("final_order_set")
    )
)


# 2. Purchase activity based on available basket details

customer_purchase_activity_df = (
    orders_with_kpis_df
    .groupBy("user_id")
    .agg(
        # Number of orders with product details
        F.count("*").alias("orders_with_product_detail"),

        # Total number of products purchased
        F.sum("basket_size").alias(
            "total_products_purchased"
        ),

        # Average basket size
        F.round(
            F.avg("basket_size"),
            2
        ).alias("avg_basket_size"),

        # Total number of reordered products
        F.sum("reordered_product_count").alias(
            "total_reordered_products"
        ),

        # Total number of new products
        F.sum("new_product_count").alias(
            "total_new_products"
        )
    )
)


# 3. Combine all customer KPIs

customer_kpis_df = (
    customer_order_activity_df

    .join(
        customer_purchase_activity_df,
        on="user_id",
        how="left"
    )

    # Overall reorder rate
    .withColumn(
        "overall_reorder_rate",
        F.round(
            F.col("total_reordered_products")
            / F.col("total_products_purchased"),
            4
        )
    )

    # Group customers by number of orders
    .withColumn(
        "customer_frequency_segment",
        F.when(
            F.col("total_order_count") <= 5,
            "occasional"
        )
        .when(
            F.col("total_order_count") <= 15,
            "regular"
        )
        .otherwise("loyal")
    )

    # Select the final columns
    .select(
        "user_id",
        "total_order_count",
        "max_order_number",
        "orders_with_product_detail",
        "total_products_purchased",
        "avg_basket_size",
        "total_reordered_products",
        "total_new_products",
        "overall_reorder_rate",
        "avg_days_between_orders",
        "final_order_set",
        "customer_frequency_segment"
    )
)

print("Customer-level KPIs created successfully.")

Customer-level KPIs created successfully.


### 4.1 Validate the Customer-Level KPIs

This check verifies that the customer KPIs are consistent.

The validation checks:

- the total number of customers
- whether each `user_id` appears only once
- whether the number of orders matches the customer's highest order number
- whether orders with product details are not greater than total orders
- whether new products and reordered products add up to total products purchased
- whether `overall_reorder_rate` is between 0 and 1
- whether the final order is either `train` or `test`

All error counts should be equal to 0.

In [0]:
# Validate the customer-level KPIs

customer_kpis_quality_df = (
    customer_kpis_df
    .agg(
        # Total number of customers
        F.count("*").alias("total_customers"),

        # Number of distinct customers
        F.countDistinct("user_id").alias(
            "distinct_customers"
        ),

        # Total orders should match the highest order number
        F.sum(
            F.when(
                F.col("total_order_count")
                != F.col("max_order_number"),
                1
            ).otherwise(0)
        ).alias("inconsistent_order_sequence"),

        # Orders with product details cannot exceed total orders
        F.sum(
            F.when(
                F.col("orders_with_product_detail")
                > F.col("total_order_count"),
                1
            ).otherwise(0)
        ).alias("invalid_orders_with_product_detail"),

        # New products + reordered products = total products
        F.sum(
            F.when(
                F.col("total_new_products")
                + F.col("total_reordered_products")
                != F.col("total_products_purchased"),
                1
            ).otherwise(0)
        ).alias("inconsistent_purchase_breakdown"),

        # Reorder rate should be between 0 and 1
        F.sum(
            F.when(
                ~F.col("overall_reorder_rate").between(0, 1),
                1
            ).otherwise(0)
        ).alias("invalid_customer_reorder_rate"),

        # Final order should be either train or test
        F.sum(
            F.when(
                ~F.col("final_order_set").isin("train", "test"),
                1
            ).otherwise(0)
        ).alias("invalid_final_order_set")
    )
)

display(customer_kpis_quality_df)

total_customers,distinct_customers,inconsistent_order_sequence,invalid_orders_with_product_detail,inconsistent_purchase_breakdown,invalid_customer_reorder_rate,invalid_final_order_set
206209,206209,0,0,0,0,0


## 5. Create Product-Level KPIs

This step creates one summary row for each product.

The product KPIs include:

- total number of purchases
- number of different customers who purchased the product
- number of reordered purchases
- number of first purchases
- product reorder rate
- average position of the product in the basket
- corresponding aisle
- corresponding department

These KPIs will help us understand product popularity and customer loyalty toward each product.

In [0]:
# Link each order to its customer

orders_users_df = (
    orders_cleaned_df
    .select(
        "order_id",
        "user_id"
    )
)


# Calculate product activity

product_activity_df = (
    order_products_cleaned_df

    # Add the customer associated with each order
    .join(
        orders_users_df,
        on="order_id",
        how="inner"
    )

    .groupBy("product_id")

    .agg(
        # Total number of times the product was purchased
        F.count("*").alias("total_purchase_count"),

        # Number of different customers who purchased the product
        F.countDistinct("user_id").alias(
            "unique_customer_count"
        ),

        # Number of reordered purchases
        F.sum(
            F.col("reordered").cast("long")
        ).alias("reordered_purchase_count"),

        # Average position of the product in the basket
        F.round(
            F.avg("add_to_cart_order"),
            2
        ).alias("avg_add_to_cart_position")
    )

    # Number of first purchases
    .withColumn(
        "first_purchase_count",
        F.col("total_purchase_count")
        - F.col("reordered_purchase_count")
    )

    # Product reorder rate
    .withColumn(
        "product_reorder_rate",
        F.round(
            F.col("reordered_purchase_count")
            / F.col("total_purchase_count"),
            4
        )
    )
)


# Add product, aisle, and department information

product_kpis_df = (
    products_cleaned_df
    .select(
        "product_id",
        "product_name",
        "aisle_id",
        "department_id"
    )

    # Keep all products from the product catalog
    .join(
        product_activity_df,
        on="product_id",
        how="left"
    )

    # Add aisle information
    .join(
        aisles_cleaned_df.select(
            "aisle_id",
            "aisle"
        ),
        on="aisle_id",
        how="left"
    )

    # Add department information
    .join(
        departments_cleaned_df.select(
            "department_id",
            "department"
        ),
        on="department_id",
        how="left"
    )

    # Products that were never purchased receive zero values
    .fillna(
        {
            "total_purchase_count": 0,
            "unique_customer_count": 0,
            "reordered_purchase_count": 0,
            "first_purchase_count": 0,
            "product_reorder_rate": 0.0
        }
    )

    # Select the final columns
    .select(
        "product_id",
        "product_name",
        "aisle_id",
        "aisle",
        "department_id",
        "department",
        "total_purchase_count",
        "unique_customer_count",
        "reordered_purchase_count",
        "first_purchase_count",
        "product_reorder_rate",
        "avg_add_to_cart_position"
    )
)

print("Product-level KPIs created successfully.")

Product-level KPIs created successfully.


### 5.1 Validate the Product-Level KPIs

This check verifies that the product KPIs are consistent.

The validation checks:

- the total number of products
- whether each `product_id` appears only once
- whether any product name is missing
- whether purchase counts contain negative values
- whether first purchases and reordered purchases add up to total purchases
- whether `product_reorder_rate` is between 0 and 1
- the total number of purchases across all products

All error counts should be equal to 0.

In [0]:
# Validate the product-level KPIs

product_kpis_quality_df = (
    product_kpis_df
    .agg(
        # Total number of products
        F.count("*").alias("total_products"),

        # Number of distinct product IDs
        F.countDistinct("product_id").alias(
            "distinct_products"
        ),

        # Products with missing names
        F.sum(
            F.when(
                F.col("product_name").isNull(),
                1
            ).otherwise(0)
        ).alias("missing_product_names"),

        # Purchase counts cannot be negative
        F.sum(
            F.when(
                (F.col("total_purchase_count") < 0)
                | (F.col("reordered_purchase_count") < 0)
                | (F.col("first_purchase_count") < 0),
                1
            ).otherwise(0)
        ).alias("invalid_purchase_counts"),

        # Reordered purchases + first purchases = total purchases
        F.sum(
            F.when(
                F.col("reordered_purchase_count")
                + F.col("first_purchase_count")
                != F.col("total_purchase_count"),
                1
            ).otherwise(0)
        ).alias("inconsistent_purchase_breakdown"),

        # Product reorder rate should be between 0 and 1
        F.sum(
            F.when(
                ~F.col("product_reorder_rate").between(0, 1),
                1
            ).otherwise(0)
        ).alias("invalid_product_reorder_rates"),

        # Total number of purchases across all products
        F.sum("total_purchase_count").alias(
            "total_purchases"
        )
    )
)

display(product_kpis_quality_df)

total_products,distinct_products,missing_product_names,invalid_purchase_counts,inconsistent_purchase_breakdown,invalid_product_reorder_rates,total_purchases
49688,49688,0,0,0,0,33819106


## 6. Create Aisle and Department KPIs

This section prepares category-level analysis for aisles and departments.

The goal is to create one summary row per aisle and one summary row per department.

The analysis will use information such as:

- number of products
- total number of purchases
- number of orders
- number of customers
- number of reordered purchases
- reorder rate
- average product position in the basket

### 6.1 Add Category Information to Purchases

This step adds customer, aisle, and department information to each purchased product.

The enriched table will be used to calculate KPIs at the aisle and department levels.

In [0]:
# Add category information to each purchased product

purchase_category_df = (
    order_products_cleaned_df

    # Add the customer associated with each order
    .join(
        orders_cleaned_df.select(
            "order_id",
            "user_id"
        ),
        on="order_id",
        how="inner"
    )

    # Add the aisle and department IDs of each product
    .join(
        products_cleaned_df.select(
            "product_id",
            "aisle_id",
            "department_id"
        ),
        on="product_id",
        how="inner"
    )

    # Add the aisle name
    .join(
        aisles_cleaned_df.select(
            "aisle_id",
            "aisle"
        ),
        on="aisle_id",
        how="left"
    )

    # Add the department name
    .join(
        departments_cleaned_df.select(
            "department_id",
            "department"
        ),
        on="department_id",
        how="left"
    )

    # Select the columns needed for category analysis
    .select(
        "order_id",
        "user_id",
        "product_id",
        "aisle_id",
        "aisle",
        "department_id",
        "department",
        "add_to_cart_order",
        "reordered"
    )
)

print("Purchase data enriched with category information successfully.")

Purchase data enriched with category information successfully.


### 6.2 Create Aisle-Level KPIs

This step creates one summary row for each aisle.

The aisle KPIs include:

- number of products in the catalog
- total number of purchases
- number of different products purchased
- number of orders containing products from the aisle
- number of customers who purchased from the aisle
- number of reordered purchases
- number of first purchases
- aisle reorder rate
- average position of aisle products in the basket

All aisles are kept, even if they have no purchase activity.

In [0]:
# Count the number of catalog products in each aisle

aisle_catalog_df = (
    products_cleaned_df
    .groupBy("aisle_id")
    .agg(
        F.countDistinct("product_id").alias(
            "catalog_product_count"
        )
    )
)


# Calculate purchase activity for each aisle

aisle_activity_df = (
    purchase_category_df

    .groupBy(
        "aisle_id",
        "aisle"
    )

    .agg(
        # Total number of purchase rows
        F.count("*").alias("total_purchase_count"),

        # Number of different products actually purchased
        F.countDistinct("product_id").alias(
            "purchased_product_count"
        ),

        # Number of orders containing at least one product from the aisle
        F.countDistinct("order_id").alias(
            "unique_order_count"
        ),

        # Number of customers who purchased from the aisle
        F.countDistinct("user_id").alias(
            "unique_customer_count"
        ),

        # Number of reordered purchases
        F.sum(
            F.col("reordered").cast("long")
        ).alias("reordered_purchase_count"),

        # Average position of aisle products in the basket
        F.round(
            F.avg("add_to_cart_order"),
            2
        ).alias("avg_add_to_cart_position")
    )

    # Number of first purchases
    .withColumn(
        "first_purchase_count",
        F.col("total_purchase_count")
        - F.col("reordered_purchase_count")
    )

    # Reorder rate for the aisle
    .withColumn(
        "reorder_rate",
        F.round(
            F.col("reordered_purchase_count")
            / F.col("total_purchase_count"),
            4
        )
    )
)


# Keep all aisles from the reference table

aisle_kpis_df = (
    aisles_cleaned_df
    .select(
        "aisle_id",
        "aisle"
    )

    .join(
        aisle_catalog_df,
        on="aisle_id",
        how="left"
    )

    .join(
        aisle_activity_df,
        on=["aisle_id", "aisle"],
        how="left"
    )

    # Aisles with no activity receive zero values
    .fillna(
        {
            "catalog_product_count": 0,
            "total_purchase_count": 0,
            "purchased_product_count": 0,
            "unique_order_count": 0,
            "unique_customer_count": 0,
            "reordered_purchase_count": 0,
            "first_purchase_count": 0,
            "reorder_rate": 0.0
        }
    )
)

print("Aisle-level KPIs created successfully.")

Aisle-level KPIs created successfully.


### 6.3 Create Department-Level KPIs

This step creates one summary row for each department.

The department KPIs include:

- number of products in the catalog
- total number of purchases
- number of different products purchased
- number of orders containing products from the department
- number of customers who purchased from the department
- number of reordered purchases
- number of first purchases
- department reorder rate
- average position of department products in the basket

All departments are kept, even if they have no purchase activity.

In [0]:
# Count the number of catalog products in each department

department_catalog_df = (
    products_cleaned_df
    .groupBy("department_id")
    .agg(
        F.countDistinct("product_id").alias(
            "catalog_product_count"
        )
    )
)


# Calculate purchase activity for each department

department_activity_df = (
    purchase_category_df

    .groupBy(
        "department_id",
        "department"
    )

    .agg(
        # Total number of purchases
        F.count("*").alias("total_purchase_count"),

        # Number of different products purchased
        F.countDistinct("product_id").alias(
            "purchased_product_count"
        ),

        # Number of orders containing products from the department
        F.countDistinct("order_id").alias(
            "unique_order_count"
        ),

        # Number of customers who purchased from the department
        F.countDistinct("user_id").alias(
            "unique_customer_count"
        ),

        # Number of reordered purchases
        F.sum(
            F.col("reordered").cast("long")
        ).alias("reordered_purchase_count"),

        # Average product position in the basket
        F.round(
            F.avg("add_to_cart_order"),
            2
        ).alias("avg_add_to_cart_position")
    )

    # Number of first purchases
    .withColumn(
        "first_purchase_count",
        F.col("total_purchase_count")
        - F.col("reordered_purchase_count")
    )

    # Reorder rate for the department
    .withColumn(
        "reorder_rate",
        F.round(
            F.col("reordered_purchase_count")
            / F.col("total_purchase_count"),
            4
        )
    )
)


# Keep all departments from the reference table

department_kpis_df = (
    departments_cleaned_df
    .select(
        "department_id",
        "department"
    )

    .join(
        department_catalog_df,
        on="department_id",
        how="left"
    )

    .join(
        department_activity_df,
        on=["department_id", "department"],
        how="left"
    )

    # Departments with no activity receive zero values
    .fillna(
        {
            "catalog_product_count": 0,
            "total_purchase_count": 0,
            "purchased_product_count": 0,
            "unique_order_count": 0,
            "unique_customer_count": 0,
            "reordered_purchase_count": 0,
            "first_purchase_count": 0,
            "reorder_rate": 0.0
        }
    )
)

print("Department-level KPIs created successfully.")

Department-level KPIs created successfully.


### 6.4 Validate Aisle and Department KPIs

This check verifies that the aisle-level and department-level KPIs are consistent.

The validation checks:

- the total number of rows
- whether each aisle or department ID appears only once
- the total number of purchases
- whether any aisle or department name is missing
- whether the number of purchased products is greater than the number of catalog products
- whether `reorder_rate` is between 0 and 1

All error counts should be equal to 0.

In [0]:
# Validate aisle-level KPIs

aisle_quality_df = (
    aisle_kpis_df
    .agg(
        # Total number of rows
        F.count("*").alias("total_rows"),

        # Number of distinct aisle IDs
        F.countDistinct("aisle_id").alias(
            "distinct_ids"
        ),

        # Total number of purchases
        F.sum("total_purchase_count").alias(
            "total_purchases"
        ),

        # Missing aisle names
        F.sum(
            F.when(
                F.col("aisle").isNull(),
                1
            ).otherwise(0)
        ).alias("missing_names"),

        # Purchased products cannot exceed catalog products
        F.sum(
            F.when(
                F.col("purchased_product_count")
                > F.col("catalog_product_count"),
                1
            ).otherwise(0)
        ).alias("inconsistent_product_counts"),

        # Reorder rate should be between 0 and 1
        F.sum(
            F.when(
                ~F.col("reorder_rate").between(0, 1),
                1
            ).otherwise(0)
        ).alias("invalid_reorder_rates")
    )

    .withColumn(
        "level",
        F.lit("aisle")
    )
)


# Validate department-level KPIs

department_quality_df = (
    department_kpis_df
    .agg(
        # Total number of rows
        F.count("*").alias("total_rows"),

        # Number of distinct department IDs
        F.countDistinct("department_id").alias(
            "distinct_ids"
        ),

        # Total number of purchases
        F.sum("total_purchase_count").alias(
            "total_purchases"
        ),

        # Missing department names
        F.sum(
            F.when(
                F.col("department").isNull(),
                1
            ).otherwise(0)
        ).alias("missing_names"),

        # Purchased products cannot exceed catalog products
        F.sum(
            F.when(
                F.col("purchased_product_count")
                > F.col("catalog_product_count"),
                1
            ).otherwise(0)
        ).alias("inconsistent_product_counts"),

        # Reorder rate should be between 0 and 1
        F.sum(
            F.when(
                ~F.col("reorder_rate").between(0, 1),
                1
            ).otherwise(0)
        ).alias("invalid_reorder_rates")
    )

    .withColumn(
        "level",
        F.lit("department")
    )
)


# Combine aisle and department validation results

category_kpis_quality_df = (
    aisle_quality_df
    .unionByName(department_quality_df)
    .select(
        "level",
        "total_rows",
        "distinct_ids",
        "total_purchases",
        "missing_names",
        "inconsistent_product_counts",
        "invalid_reorder_rates"
    )
)

display(category_kpis_quality_df)

level,total_rows,distinct_ids,total_purchases,missing_names,inconsistent_product_counts,invalid_reorder_rates
aisle,134,134,33819106,0,0,0
department,21,21,33819106,0,0,0


## 7. Create Time-Based KPIs

This section analyzes order activity using the available time information in the dataset.

Since the dataset does not contain real calendar dates, the analysis is based on:

- day of the week (`order_dow`)
- hour of the day (`order_hour_of_day`)
- time period (`order_time_period`)

For each time group, the analysis calculates:

- total number of orders
- number of different customers
- total number of products purchased
- average basket size
- number of reordered products
- number of new products
- average number of days between orders
- reorder rate

In [0]:
def create_temporal_kpis(grouping_column):
    """
    Create order and purchase KPIs
    for a given time dimension.
    """

    return (
        orders_with_kpis_df

        # Group by day, hour, or time period
        .groupBy(grouping_column)

        .agg(
            # Total number of orders
            F.count("*").alias("total_order_count"),

            # Number of different customers
            F.countDistinct("user_id").alias(
                "unique_customer_count"
            ),

            # Total number of products purchased
            F.sum("basket_size").alias(
                "total_purchase_count"
            ),

            # Average basket size
            F.round(
                F.avg("basket_size"),
                2
            ).alias("avg_basket_size"),

            # Number of reordered products
            F.sum("reordered_product_count").alias(
                "reordered_purchase_count"
            ),

            # Number of new products
            F.sum("new_product_count").alias(
                "new_product_count"
            ),

            # Average number of days between orders
            F.round(
                F.avg("days_since_prior_order"),
                2
            ).alias("avg_days_between_orders")
        )

        # Reorder rate based on the number of products
        .withColumn(
            "reorder_rate",
            F.round(
                F.col("reordered_purchase_count")
                / F.col("total_purchase_count"),
                4
            )
        )

        .orderBy(grouping_column)
    )

### 7.2 Create KPIs by Day of Week

This analysis groups orders by `order_dow`, where the values range from 0 to 6.

For each day, the table shows:

- total number of orders
- number of different customers
- total number of products purchased
- average basket size
- number of reordered products
- number of new products
- average number of days between orders
- reorder rate

This helps compare customer activity across the different days of the week.

In [0]:

daily_kpis_df = create_temporal_kpis(
    "order_dow"
)

display(daily_kpis_df)

order_dow,total_order_count,unique_customer_count,total_purchase_count,avg_basket_size,reordered_purchase_count,new_product_count,avg_days_between_orders,reorder_rate
0,585237,149694,6533692,11.16,3831900,2701792,11.63,0.5865
1,576377,155758,5871834,10.19,3544661,2327173,11.17,0.6037
2,458074,144571,4378360,9.56,2582006,1796354,11.03,0.5897
3,428087,139743,3998498,9.34,2344277,1654221,10.62,0.5863
4,417171,136719,3942696,9.45,2330620,1612076,10.38,0.5911
5,443388,138364,4386443,9.89,2613888,1772555,10.37,0.5959
6,437749,132612,4707583,10.75,2708008,1999575,11.3,0.5752


### 7.3 Create KPIs by Hour of Day

This analysis groups orders by `order_hour_of_day`, with values from 0 to 23.

For each hour, the table shows:

- total number of orders
- number of different customers
- total number of products purchased
- average basket size
- number of reordered products
- number of new products
- average number of days between orders
- reorder rate

This helps identify the hours with the highest and lowest order activity.

In [0]:

hourly_kpis_df = create_temporal_kpis(
    "order_hour_of_day"
)

display(hourly_kpis_df)

order_hour_of_day,total_order_count,unique_customer_count,total_purchase_count,avg_basket_size,reordered_purchase_count,new_product_count,avg_days_between_orders,reorder_rate
0,22224,15993,228031,10.26,129003,99028,11.77,0.5657
1,12103,9189,121412,10.03,67766,53646,12.0,0.5581
2,7375,5851,72660,9.85,40368,32292,11.8,0.5556
3,5343,4410,53759,10.06,30132,23627,12.03,0.5605
4,5393,4469,55714,10.33,31889,23825,12.03,0.5724
5,9374,7141,91909,9.8,55933,35976,11.44,0.6086
6,29913,18948,302642,10.12,192798,109844,10.64,0.637
7,90032,46020,928239,10.31,598407,329832,10.52,0.6447
8,174664,76750,1787359,10.23,1130174,657185,10.56,0.6323
9,252529,100019,2550569,10.1,1580305,970264,10.66,0.6196


### 7.4 Create KPIs by Time Period

This analysis groups orders by `order_time_period`.

The day is divided into four periods:

- `morning`
- `afternoon`
- `evening`
- `night`

For each period, the table shows:

- total number of orders
- number of different customers
- total number of products purchased
- average basket size
- number of reordered products
- number of new products
- average number of days between orders
- reorder rate

This helps compare customer activity across different periods of the day.

In [0]:


time_period_kpis_df = create_temporal_kpis(
    "order_time_period"
)

display(time_period_kpis_df)

order_time_period,total_order_count,unique_customer_count,total_purchase_count,avg_basket_size,reordered_purchase_count,new_product_count,avg_days_between_orders,reorder_rate
afternoon,1359023,195315,13749997,10.12,7979643,5770354,11.1,0.5803
evening,717903,160645,7067571,9.84,4084793,2982778,11.0,0.578
morning,1117598,177445,11388324,10.19,6960387,4427937,10.8,0.6112
night,151559,64423,1613214,10.64,930537,682677,11.05,0.5768


### 7.5 Validate the Time-Based KPIs

This check verifies that the time-based KPI tables are consistent.

The validation checks:

- whether the expected number of time groups is present
- the total number of orders found in each time analysis
- the total number of purchases found in each time analysis
- whether `reorder_rate` stays between 0 and 1

The expected number of groups is:

- 7 groups for day of week
- 24 groups for hour of day
- 4 groups for time period

All invalid reorder-rate counts should be equal to 0.

In [0]:
# Validate the time-based KPIs

def validate_temporal_kpis(
    dataframe,
    level_name,
    expected_group_count
):
    """
    Validate the number of groups, total volumes,
    and consistency of reorder rates.
    """

    return (
        dataframe
        .agg(
            # Number of time groups
            F.count("*").alias("group_count"),

            # Total number of orders
            F.sum("total_order_count").alias(
                "total_orders"
            ),

            # Total number of purchases
            F.sum("total_purchase_count").alias(
                "total_purchases"
            ),

            # Reorder rate should be between 0 and 1
            F.sum(
                F.when(
                    ~F.col("reorder_rate").between(0, 1),
                    1
                ).otherwise(0)
            ).alias("invalid_reorder_rates")
        )

        # Name of the time level
        .withColumn(
            "level",
            F.lit(level_name)
        )

        # Expected number of groups
        .withColumn(
            "expected_group_count",
            F.lit(expected_group_count)
        )
    )


# Validate day-of-week KPIs

daily_quality_df = validate_temporal_kpis(
    daily_kpis_df,
    "day_of_week",
    7
)


# Validate hourly KPIs

hourly_quality_df = validate_temporal_kpis(
    hourly_kpis_df,
    "hour_of_day",
    24
)


# Validate time-period KPIs

time_period_quality_df = validate_temporal_kpis(
    time_period_kpis_df,
    "time_period",
    4
)


# Combine all validation results

temporal_kpis_quality_df = (
    daily_quality_df
    .unionByName(hourly_quality_df)
    .unionByName(time_period_quality_df)

    .select(
        "level",
        "group_count",
        "expected_group_count",
        "total_orders",
        "total_purchases",
        "invalid_reorder_rates"
    )
)

display(temporal_kpis_quality_df)

level,group_count,expected_group_count,total_orders,total_purchases,invalid_reorder_rates
day_of_week,7,7,3346083,33819106,0
hour_of_day,24,24,3346083,33819106,0
time_period,4,4,3346083,33819106,0


## 8. Save the Analysis Tables

This step saves all KPI DataFrames as permanent Delta tables in `workspace.analysis_data`.

The following analysis tables are created:

- `order_kpis`
- `customer_kpis`
- `product_kpis`
- `aisle_kpis`
- `department_kpis`
- `daily_kpis`
- `hourly_kpis`
- `time_period_kpis`

A timestamp is also added to each table to record when the analysis results were created.

These saved tables will be used later for analysis, visualization, and dashboard creation.

In [0]:
# Map each table name to its corresponding KPI DataFrame

analysis_tables = {
    "order_kpis": orders_with_kpis_df,
    "customer_kpis": customer_kpis_df,
    "product_kpis": product_kpis_df,
    "aisle_kpis": aisle_kpis_df,
    "department_kpis": department_kpis_df,
    "daily_kpis": daily_kpis_df,
    "hourly_kpis": hourly_kpis_df,
    "time_period_kpis": time_period_kpis_df
}


# Save each DataFrame as a Delta table in workspace.analysis_data

for table_name, dataframe in analysis_tables.items():

    # Build the full table name
    full_table_name = (
        f"{catalog_name}."
        f"{analysis_schema_name}."
        f"{table_name}"
    )

    # Add the date and time when the KPI table was created
    dataframe_to_save = (
        dataframe
        .withColumn(
            "_analysis_processed_at",
            F.current_timestamp()
        )
    )

    # Create or replace the Delta table
    (
        dataframe_to_save.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(full_table_name)
    )

    print(f"Saved table: {full_table_name}")


print("All analysis tables were saved successfully.")

Saved table: workspace.analysis_data.order_kpis
Saved table: workspace.analysis_data.customer_kpis
Saved table: workspace.analysis_data.product_kpis
Saved table: workspace.analysis_data.aisle_kpis
Saved table: workspace.analysis_data.department_kpis
Saved table: workspace.analysis_data.daily_kpis
Saved table: workspace.analysis_data.hourly_kpis
Saved table: workspace.analysis_data.time_period_kpis
All analysis tables were saved successfully.


## 9. Final Verification of Analysis Tables

This final check confirms that all KPI tables were successfully saved in `workspace.analysis_data`.

The table list is displayed to verify that the expected analysis tables are available before moving to data analysis and visualization.

In [0]:

display(
    spark.sql(
        f"""
        SHOW TABLES IN
        {catalog_name}.{analysis_schema_name}
        """
    )
    .orderBy("tableName")
)

database,tableName,isTemporary
analysis_data,aisle_kpis,false
analysis_data,customer_kpis,false
analysis_data,daily_kpis,false
analysis_data,department_kpis,false
analysis_data,hourly_kpis,false
analysis_data,order_kpis,false
analysis_data,product_kpis,false
analysis_data,time_period_kpis,false


### 9.1 Verify Saved Table Row Counts

This check confirms how many rows were saved in each analysis table.

It helps verify that the Delta tables contain the expected amount of data after being written to `workspace.analysis_data`.

The result shows one row for each analysis table with its saved row count.

In [0]:

saved_table_counts = []

for table_name in analysis_tables.keys():

    full_table_name = (
        f"{catalog_name}."
        f"{analysis_schema_name}."
        f"{table_name}"
    )

    row_count = (
        spark.table(full_table_name)
        .count()
    )

    saved_table_counts.append(
        (table_name, row_count)
    )


saved_table_counts_df = spark.createDataFrame(
    saved_table_counts,
    [
        "table_name",
        "row_count"
    ]
)

display(
    saved_table_counts_df
    .orderBy("table_name")
)

table_name,row_count
aisle_kpis,134
customer_kpis,206209
daily_kpis,7
department_kpis,21
hourly_kpis,24
order_kpis,3346083
product_kpis,49688
time_period_kpis,4
